# token_analysis
Tokenization analysis for the test set of abstracts using `llm_from_scratch.utils.token_analysis`.
Note that the test set was used in a lot of the domain-adaptive pretraining, which followed just loading the publically available weights for gpt2-small as trained by OpenAI.

In [1]:
# Optional Google Drive mount (Colab only)
import os
from pathlib import Path

def in_colab() -> bool:
    return "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ or "google.colab" in str(getattr(__import__("sys"), "modules", {}))

IN_COLAB = in_colab()

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        print("Mounted Google Drive at /content/drive")
    except Exception as e:
        print(f"Could not mount Google Drive: {e}")
else:
    print("Not running in Colab; skipping Drive mount.")


Not running in Colab; skipping Drive mount.


In [2]:
# Resolve project root and paths
import sys

env_root = os.environ.get("LLM_PROJECT_ROOT", "").strip()
default_colab_root = Path("/content/drive/MyDrive/llm-from-scratch-drive")

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists():
            return p
    return start

if env_root:
    PROJECT_ROOT = Path(env_root).expanduser().resolve()
elif default_colab_root.exists():
    PROJECT_ROOT = default_colab_root.resolve()
else:
    PROJECT_ROOT = find_repo_root(Path.cwd().resolve())

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


CWD: /home/markb/llm-from-scratch/notebooks
PROJECT_ROOT: /home/markb/llm-from-scratch
SRC_DIR: /home/markb/llm-from-scratch/src
DATA_DIR: /home/markb/llm-from-scratch/data


In [3]:
# Imports and tokenizer setup
from collections import Counter

import pandas as pd
import tiktoken

from llm_from_scratch.utils import token_analysis as ta

ta.tokenizer = tiktoken.get_encoding("gpt2")
print("Tokenizer ready: gpt2")


Tokenizer ready: gpt2


In [4]:
# Load the test abstract set - because it is relatively
filenm = "pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_test_abstracts.txt"
text_path = DATA_DIR / filenm
with open(text_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Text path:", text_path)
print("Characters:", len(text))
print("Lines:", text.count("\n") + 1)
print("Preview:\n")
print(text[:500])


Text path: /home/markb/llm-from-scratch/data/pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_test_abstracts.txt
Characters: 7267986
Lines: 4473
Preview:

BACKGROUND: Breast cancer in the elderly is associated with high recurrence and death rates, due mostly to undertreatment. Human epidermal growth factor receptor type 2 (HER2) overexpression is infrequent in older patients. Trastuzumab-based chemotherapy is often withheld from elderly patients because of its cardiotoxicity. PATIENTS AND METHODS: Medical records of consecutive HER2-positive breast cancer patients aged >/=70 years old treated between 2005 and 2010 in the participating centers were


In [5]:
# Count token ids and inspect top tokens
token_id_counts = ta.count_token_ids([text])
top_tokens_df = ta.top_n_token_df(token_id_counts, n=100)

print("Unique token ids:", len(token_id_counts))
print("Total tokens:", sum(token_id_counts.values()))
print(top_tokens_df)


Unique token ids: 19423
Total tokens: 1679880
    tokenid decoded_repr  count  fraction_total
0        13          '.'  59144        0.035207
1        11          ','  48083        0.028623
2        12          '-'  46175        0.027487
3       290       ' and'  39109        0.023281
4       286        ' of'  36991        0.022020
..      ...          ...    ...             ...
95     1172      ' prog'   2081        0.001239
96      407       ' not'   2067        0.001230
97    24561   'positive'   2061        0.001227
98      459        'ast'   2057        0.001224
99       20          '5'   2056        0.001224

[100 rows x 4 columns]


In [6]:
# Group token counts after left-strip (e.g., ' the' and 'the')
decoded_token_counts = Counter({ta.decode_id(tid): count for tid, count in token_id_counts.items()})
grouped_counts = ta.group_counts_by_lstrip(decoded_token_counts)

grouped_df = pd.DataFrame(grouped_counts.most_common(40), columns=["token_text", "count"])
grouped_df["fraction_total"] = grouped_df["count"] / grouped_df["count"].sum()
grouped_df


,token_text,count,fraction_total
0,.,59716,0.098099
1,",",48121,0.079051
2,-,46545,0.076462
3,and,39461,0.064825
4,of,37208,0.061124
5,the,34029,0.055901
6,in,31937,0.052465
7,(,30648,0.050347
8,to,17131,0.028142
9,with,16878,0.027726


In [7]:
# Build word-like units from GPT token boundaries
units = list(ta.wordlike_units(text))
unit_counts = Counter(units)

print("Total word-like units:", len(units))
print("Unique word-like units:", len(unit_counts))

unit_df = pd.DataFrame(unit_counts.most_common(40), columns=["unit", "count"])
unit_df["fraction_total"] = unit_df["count"] / len(units)
unit_df


Total word-like units: 1035356
Unique word-like units: 77446


,unit,count,fraction_total
0,and,38700,0.037378
1,of,36983,0.035720
2,the,33962,0.032802
3,in,25298,0.024434
4,to,16875,0.016299
5,with,16832,0.016257
6,a,12783,0.012346
7,was,10187,0.009839
8,for,10086,0.009742
9,were,9066,0.008756


In [8]:
# Top words using simple split on whitespace, commas, colons, semicolons, and periods
top_simple_words_df = ta.top_n_words_simple_split(text, n=40)
top_simple_words_df


,word,count,fraction_total
0,and,38777,0.036766
1,of,36987,0.035069
2,the,33962,0.032201
3,in,25304,0.023992
4,to,16884,0.016009
5,with,16836,0.015963
6,a,12784,0.012121
7,cancer,10616,0.010066
8,was,10195,0.009666
9,for,10088,0.009565
